In [1]:
! pip install ollama

In [2]:
import ollama
language_model = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest"
embedding_model ="hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest"

In [8]:
datasets = []
with open("cat-facts.txt", "r", encoding="utf-8") as f:
    datasets = f.readlines()

print(f"Loaded {len(datasets)} cat facts")

Loaded 150 cat facts


In [9]:
VECTOR_DB = []


In [10]:
def add_chunk_to_vector_db(chunk):
    embedding = ollama.embed(model=embedding_model, input=chunk)['embedding'][0]
    VECTOR_DB.append((chunk, embedding))

In [11]:
def add_chunk_to_vector_db(chunk):
    # Generate embedding for the chunk
    embeddings = ollama.embed(model=embedding_model, input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk,embeddings))

In [12]:
for i,chunk in enumerate(datasets):
    add_chunk_to_vector_db(chunk)
    print(f"Added chunk {i+1}/{len(datasets)} to vector database")

Added chunk 1/150 to vector database
Added chunk 2/150 to vector database
Added chunk 3/150 to vector database
Added chunk 4/150 to vector database
Added chunk 5/150 to vector database
Added chunk 6/150 to vector database
Added chunk 7/150 to vector database
Added chunk 8/150 to vector database
Added chunk 9/150 to vector database
Added chunk 10/150 to vector database
Added chunk 11/150 to vector database
Added chunk 12/150 to vector database
Added chunk 13/150 to vector database
Added chunk 14/150 to vector database
Added chunk 15/150 to vector database
Added chunk 16/150 to vector database
Added chunk 17/150 to vector database
Added chunk 18/150 to vector database
Added chunk 19/150 to vector database
Added chunk 20/150 to vector database
Added chunk 21/150 to vector database
Added chunk 22/150 to vector database
Added chunk 23/150 to vector database
Added chunk 24/150 to vector database
Added chunk 25/150 to vector database
Added chunk 26/150 to vector database
Added chunk 27/150 to

In [13]:
#Defining a function to calculate cosine similarity between two vectors
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = sum(a ** 2 for a in vec1) ** 0.5
    magnitude2 = sum(b ** 2 for b in vec2) ** 0.5
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)

In [14]:
def retriever(query, top_k=3):
    query_embedding = ollama.embed(model=embedding_model, input=query)['embedding'][0]
    similarities = []
    for chunk, embedding in VECTOR_DB:
        sim = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [chunk for chunk, _ in similarities[:top_k]]

In [15]:
query = "What is the average lifespan of a cat?"

In [20]:
result = retriever(query)
print("Top relevant chunks:")
for i,chunk in enumerate(result):
  print(f"{i+1}, {chunk.strip()}")

Top relevant chunks:
1, When well treated, a cat can live twenty or more years but the average life span of a domestic cat is 14 years.
2, The oldest cat on record was Crème Puff from Austin, Texas, who lived from 1967 to August 6, 2005, three days after her 38th birthday. A cat typically can live up to 20 years, which is equivalent to about 96 human years.
3, On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.
